# Workshop on Statistical Genetics and Genetic Epidemiology STAGE-Quebec
## Theme 2 - Molecular Phenotypes in Genetic Epidemiology

By Marc-André Legault (Université de Montréal) and Qihuang Zhang (McGill University)

**July 28 and 29, 2026**

---

## Understanding TWAS Results: Interpreting Gene-Disease Associations

### Analysis Overview

In our previous analysis, we conducted a Transcriptome-Wide Association Study (TWAS) of Type 2 Diabetes (T2D) across 7 GTEx tissues using gene expression prediction models from PredictDB and the S-PrediXcan framework. This notebook focuses on the interpretation of these results and their biological implications.

### Learning Objectives

By the end of this notebook, you will be able to:
- Apply appropriate statistical corrections for multiple testing in TWAS
- Interpret tissue-specific gene expression associations
- Visualize and assess the strength of gene-disease relationships
- Connect TWAS findings back to underlying genetic variants

### Analysis Strategy

We will systematically evaluate our results using different statistical approaches. First, we calculate appropriate P-value thresholds for significance testing and summarize our findings across tissues.

**Research Question:** How many genes show significant associations with T2D, and how does this depend on our statistical approach?

| Tissue               | N tested genes | Global Bonferroni | Tissue Bonferroni |
| :------------------: | -------------- | ----------------- | ----------------- |
| Adipose subcutaneous |                |                   |                   |
| ...                  |                |                   |                   |

In [ ]:
library(ggplot2)

In [ ]:
tissues <- c(
    "en_Adipose_Subcutaneous",
    "en_Artery_Coronary",
    "en_Brain_Cortex",
    "en_Liver",
    "en_Muscle_Skeletal",
    "en_Pancreas",
    "en_Whole_Blood"
)

df <- data.frame()
for (tissue in tissues) {
    cur <- read.csv(paste0("/workshop/local/results/twas_", tissue, ".csv"))
    df <- rbind(df, data.frame(tissue = tissue, n_tested_genes = nrow(cur)))
}

df$global_bonf <- 0.05 / sum(df$n_tested_genes)
df$tissue_bonf <- 0.05 / df$n_tested_genes
df

### Statistical Threshold Interpretation

The results above demonstrate an important characteristic of our analysis scale. Even with a conservative global Bonferroni correction across **all genes in all tissues**, our significance threshold remains more lenient than the standard genome-wide significance threshold (5 × 10⁻⁸). This reflects a key advantage of TWAS: **we test far fewer hypotheses** than in a traditional GWAS, which increases our statistical power to detect true associations.

**Key Observations:**
- The number of testable genes varies substantially across tissues (3,765 in liver vs. 8,626 in subcutaneous adipose)
- This variation directly affects our tissue-specific Bonferroni thresholds
- More testable genes result in stricter significance thresholds for that tissue

We will now compare different multiple testing approaches and examine how they affect our discovery of significant genes. We examine three approaches: global Bonferroni, tissue-specific Bonferroni, and the False Discovery Rate (FDR) method.

<div class="alert alert-success">
<strong>Note:</strong> The number of testable genes varies across tissues due to differences in the requirements for building reliable gene expression prediction models and the challenges of measuring gene expression in different tissue types.
</div>

In [ ]:
n_sig <- data.frame()
sig_genes_data <- data.frame()
sig_genes <- list()

for (tissue in tissues) {
    cur <- read.csv(paste0("/workshop/local/results/twas_", tissue, ".csv"))

    sig_global_bonf <- cur$pvalue <= 1.174205e-6
    sig_tissue_bonf <- cur$pvalue <= 0.05 / nrow(cur)
    q <- p.adjust(cur$pvalue, method = "BH")
    sig_fdr <- q <= 0.05
    
    sig_genes_data_cur <- cur[sig_fdr, c("gene_name", "pvalue", "effect_size", "n_snps_in_model", "n_snps_used", "pred_perf_pval")]
    sig_genes_data_cur$tissue <- tissue
    sig_genes_data <- rbind(sig_genes_data, sig_genes_data_cur)
    
    sig_genes[[tissue]] <- list(
        sig_global_bonf = cur[sig_global_bonf, "gene_name"],
        sig_tissue_bonf = cur[sig_tissue_bonf, "gene_name"],
        sig_fdr = cur[sig_fdr, "gene_name"]
    )
    
    n_sig <- rbind(n_sig, data.frame(
        tissue = tissue,
        sig_global_bonf = sum(sig_global_bonf),
        sig_tissue_bonf = sum(sig_tissue_bonf),
        sig_fdr = sum(sig_fdr)
    ))
}
n_sig

### Multiple Testing Correction Comparison

The table above demonstrates how different statistical approaches affect our gene discovery. This comparison illustrates a fundamental trade-off in statistical genetics: **controlling false positives vs. maintaining discovery power**.

<div class="alert alert-info">
<strong>Analysis Questions:</strong>
<ol>
    <li><strong>Correction stringency:</strong> Which correction is most strict? Which is least strict?</li>
    <li><strong>Appropriate method for TWAS:</strong> Which approach is most suitable for TWAS analysis? Consider:<ul>
        <li>Independence of tests across tissues</li>
        <li>Independence of tests across genes within tissues</li>
        <li>Importance of controlling family-wise error rate vs. false discovery rate</li></ul>
    </li>
    <li><strong>Method assumptions:</strong><ul>
        <li><em>Global Bonferroni:</em> Controls probability of any false positive across all tests</li>
        <li><em>Tissue-specific Bonferroni:</em> Controls false positives within each tissue separately</li>
        <li><em>FDR (Benjamini-Hochberg):</em> Controls expected proportion of false discoveries</li></ul>
    </li>
</ol>
</div>

Enter your answer here

In [ ]:
cat("Significant genes at FDR <= 5% across tissues:\n\n")
for (tissue in tissues) {
    cat(paste0(
        "\t'", tissue, "': ",
        paste0(sig_genes[[tissue]]$sig_fdr, collapse = ", "),
        "\n"
    ))
}

### Tissue-Specific Gene Expression Patterns

The gene list above reveals notable patterns of tissue specificity in T2D associations. These patterns provide insights into disease biology:

<div class="alert alert-info">
<strong>Biological Observations:</strong>
<br><br>
<strong>Pan-tissue Association:</strong> The genetically predicted expression of <em>AP3S2</em> is associated with T2D across <strong>all</strong> tissues examined. This suggests AP3S2 may play a fundamental role in T2D pathogenesis that transcends tissue boundaries. It is also possible that eQTLs are shared across different tissues. Going back to GTEx can help inform the tissue specificity of eQTLs.
<br><br>
<strong>Tissue-Restricted Effects:</strong> Other genes show more limited patterns. For example, <em>FTO</em> shows a significant association only in skeletal muscle tissue.
<br><br>
<strong>Research Question:</strong> What biological mechanisms could explain why FTO's association with T2D is specific to skeletal muscle?
<br><br>
<em>Possible explanations:</em>
<ul>
<li><strong>True tissue specificity:</strong> FTO's role in T2D biology may be genuinely muscle-specific</li>
<li><strong>Expression heritability:</strong> FTO expression might only be sufficiently heritable in skeletal muscle to build a reliable prediction model</li>
<li><strong>Statistical power:</strong> Greater sample sizes or stronger eQTL effects in muscle tissue may have provided more power for detection</li>
<li><strong>Measurement quality:</strong> Gene expression measurement or prediction accuracy might vary across tissues</li>
</ul>
    
Take some time to discuss these results as a group and enter your comments below.
</div>


Enter your group notes here.


Understanding these patterns is important for interpreting TWAS results and designing follow-up functional studies. We will now visualize these associations to better understand their directionality and strength.

In [ ]:
source("/workshop/utilities/gene_tissue_dotplot.R")
options(repr.plot.width = 10, repr.plot.height = 5)
options(repr.plot.res = 200)
plot_gene_tissue_dots_advanced(sig_genes_data)

### Interpreting Association Patterns: The Dot Plot

The visualization above provides information for interpreting our TWAS results and their potential therapeutic implications. Each dot represents a gene-tissue combination, with key features indicating:

- **Color intensity:** Significance level (darker = more significant)
- **Dot size:** Magnitude of the association (larger = stronger effect)
- **Color hue:** Direction of association (red = positive, blue = negative)

**Effect Direction Interpretation:**
- **Positive association (red):** Higher genetically predicted gene expression → Higher T2D risk
- **Negative association (blue):** Higher genetically predicted gene expression → Lower T2D risk

<div class="alert alert-info">
<strong>Therapeutic Target Evaluation:</strong>
<br><br>
If we assume these associations reflect causal relationships, which genes would be the most promising therapeutic targets for preventing T2D? In which tissues should efforts be focused?
<br><br>
<strong>Consideration:</strong> For drug development, we typically target genes where:
<ul>
<li>Higher expression is <em>positively</em> associated with disease risk (amenable to inhibition)</li>
<li>Therapeutic delivery in the tissue of interest will be possible</li>
<li>The effect size is <em>substantial</em> enough to provide meaningful therapeutic benefit</li>
</ul>
<br>
<strong>Example:</strong> Based on these criteria, genes like <em>WFS1</em> in adipose tissue or <em>ANK1</em> in skeletal muscle may represent promising targets for therapeutic intervention.
</div>

Next, we examine the genetic architecture underlying these associations by investigating how TWAS signals relate to the underlying GWAS variants.

In [ ]:
# We define the WFS1 region using data from Ensembl.
# See: https://useast.ensembl.org/Homo_sapiens/Gene/Summary?db=core;g=ENSG00000109501;r=4:6269849-6303265
# We added +/- 100kb padding.
wfs1_region <- list(
    chrom = "chr4",
    start = 6269849 - 100000,
    end = 6303265 + 100000
)

get_predictdb_and_gwas_data <- function(region, gene_name, tissue) {
    # We read the GWAS summary statistics, but keep only the information for
    # variants in the region of interest.
    gwas <- read.csv(
        paste0(
            "/workshop/data/PrediXcan/gwas_harmonized/harmonized_",
            tissue,
            ".tsv.gz"
        ), sep="\t"
    )
    gwas_region <- gwas[
        (gwas$chromosome == region$chrom) &
        (region$start <= gwas$base_pair_location) &
        (gwas$base_pair_location <= region$end),
    ]
    rm(gwas)

    # We will extract the rsid and weights of variants included in the
    # gene expression predictive model.
    db_filename = paste0("/workshop/data/PrediXcan/predictdb/", tissue, ".db")
        
    model <- system(paste0(
        "sqlite3 ",
        db_filename,
        " \"select rsid, weight from weights w, extra e where e.gene=w.gene and e.genename='",
        gene_name, "';\""
    ), intern=T)

    if (length(model) == 0) {
        stop("Can't find PredictDB data for gene.")
    }

    model <- data.frame(do.call(rbind, strsplit(model, "\\|")))
    colnames(model) <- c("rsid", "weight")
    model$weight <- as.numeric(model$weight)

    # We create indicator variables to test whether a variant is in the model
    # and we set a default value for the weights for visualization purposes.
    gwas_region$in_model <- gwas_region$rsid_x %in% model$rsid

    gwas_region <- merge(gwas_region, model, by.x = "rsid_x", by.y = "rsid", all.x = T)
    min_abs_weight = min(abs(na.omit(gwas_region$weight)))
    gwas_region$weight <- ifelse(
        is.na(gwas_region$weight),
        0.8 * min_abs_weight,
        gwas_region$weight
    )
    
    list(
        model = model,
        gwas_region = gwas_region
    )
}

plot_region <- function(region, gene_name, tissue) {
    data <- get_predictdb_and_gwas_data(region, gene_name, tissue)
    
    options(repr.plot.width = 10, repr.plot.height = 4)
    options(repr.plot.res = 200)
    
    plot <- ggplot(data$gwas_region, aes(x = base_pair_location, y = neg_log_10_p_value,
                      color = in_model, size = abs(weight))) +
        geom_point(alpha = 0.7) +
        scale_color_manual(values = c("FALSE" = "#222222", "TRUE" = "#2E86C1")) +
        labs(x = paste0("Base pair location (", region$chr, " ,GRCh38)"),
             y = "-log10(P)",
             color = paste0("In ", gene_name, " model")) +
        theme_minimal()
    
    plot
}

In [ ]:
plot_region(wfs1_region, "WFS1", "en_Adipose_Subcutaneous")

### Connecting TWAS to GWAS: Understanding the Underlying Genetic Architecture

The regional plot above is a tool for validating and interpreting our TWAS findings. It illustrates the relationship between our gene expression association and the underlying genetic variants that drive it.

**Plot Elements:**
- **Gray dots:** All genetic variants in the *WFS1* region tested in the T2D GWAS
- **Blue dots:** Variants included in the *WFS1* expression prediction model for adipose tissue
- **Dot size:** Proportional to the variant's weight in the prediction model
- **Y-axis:** Strength of association with T2D (-log₁₀ P-value)

<div class="alert alert-info">
<strong>Interpretation</strong>
<br><br>
1. <strong>TWAS validation:</strong>
   <br><em>Examine:</em> Do the blue dots (model variants) show stronger T2D associations than random variants in the region?
<br><br>
2. <strong>Mechanistic inference:</strong>
   <br><em>Interpretation:</em> The variants driving *WFS1* expression are the same variants associated with T2D risk - this supports a model where genetic variants → gene expression → disease risk.
</div>

**Further Analysis:**
Experiment with the commented code below to explore *WFS1* expression models in other tissues, or try visualizing other significant genes from our analysis. 

**Research Questions to Explore:**
- Do different tissues use different sets of variants to predict *WFS1* expression?
- Are tissue-specific prediction models supported by tissue-specific GWAS signals?
- How do genes with tissue-specific associations (like *FTO*) compare to genes with broad associations (like *AP3S2*)?

This analysis is essential for distinguishing likely causal relationships from statistical artifacts and for prioritizing genes for functional follow-up studies.

In [ ]:
# plot_region(wfs1_region, "WFS1", "en_Whole_Blood")
# plot_region(wfs1_region, "WFS1", "en_Pancreas")
# plot_region(wfs1_region, "WFS1", "en_Muscle_Skeletal")

### Next steps

In the [next notebook](./2a-FUSION.ipynb), we will use TWAS FUSION, a second TWAS algorithm to see how it works and compares to S-PrediXcan.